### Ingesting jfjelstut datasets from staging : This source contains men and women fifa worldcup from 1930 to 2022

In [0]:
# Databricks notebook source : dlt_bronze_jfjelstul
# ═══════════════════════════════════════════════════════════════════════════════
# Factory metadata-driven : staging (Volume) -> Bronze (Schema)
# Contract : execute only pipeline pl_ingesting_bronze_jfjelstul
# ═══════════════════════════════════════════════════════════════════════════════
import dlt
import json
import os
from pyspark.sql import functions as F

# 1. Environment Context
ENV         = spark.conf.get("fifa.env", "dev")
CATALOG     = spark.conf.get("fifa.catalog", "fifa_platform")
VOLUME_PATH = spark.conf.get("fifa.volume_path",
    "/Volumes/fifa_worldcup_data/staging/staging_fifa_data/raw_data"
)

# 2. Reading Config File : METADATA-DRIVEN
def find_repo_root(marker: str = "configs") -> str:
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.isdir(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(
        f"Folder '{marker}' not found when tracing back from {os.getcwd()}"
    )
    
CONFIG_PATH = spark.conf.get(
    "fifa.bronze_config_path",
    os.path.join(find_repo_root(), "configs", "config_bronze_jfjelstul.json")
)
with open(CONFIG_PATH) as f:
    _raw = json.load(f)

DATASETS = [d for d in _raw["datasets"] if d.get("active", True)]

# Sanity Check : Only for test purpose, not promoted in PROD
assert len(DATASETS) > 0, (
    f"config_bronze_jfjelstul.json contains no active datasets "
    f"(path read : {CONFIG_PATH})"
)


# 3. The factory : One function to build each tables from the dataset
def make_bronze_table(ds: dict):
    """Declare a Bronze streaming table for a dataset from the configuration.
    The ds argument sets the value (bypassing Python’s late binding)."""
    table_name = ds["target_table"]
    src_path   = f"{VOLUME_PATH}/{ds['staging_subpath']}"
    schema_loc = f"{VOLUME_PATH}/_schemas/jfjelstul/{table_name}" # folder _schemas/jfjelstul/ will be automatically created under your volume

    @dlt.table(
        name=table_name,
        comment=(
            f"Bronze · {ds['id']} · repo {ds['source_repo']} · "
            f"raw data + audit's columns · env {ENV}"
        ),
        table_properties={
            "quality/flavour":     "bronze",
            "fifa.source_repo":  ds["source_repo"],
            "fifa.dataset_id":   ds["id"],
            "fifa.env":          ENV
        }
    )
    def _bronze():
        return (
            spark.readStream
                 .format("cloudFiles")
                 .option("cloudFiles.format", "csv")
                 .option("header", "true")
                 .option("cloudFiles.inferColumnTypes", "true")
                 .option("cloudFiles.schemaLocation", schema_loc)
                 .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                 .load(src_path)
                 .withColumn("_source_file", F.col("_metadata.file_path"))
                 .withColumn("_ingested_at", F.from_utc_timestamp(F.current_timestamp(), "America/New_York")) # Eastern Standard Time
                 .withColumn("_file_size_MB", F.round(F.col("_metadata.file_size") / 1_048_576).cast("int"))
                 .withColumn("_source_repo", F.lit(ds["source_repo"]))
        )
    return _bronze

# 4. Loop : N statements from config file
for ds in DATASETS:
    make_bronze_table(ds)